# Probing Jailbreak Brittleness: Probe Analysis

This notebook develops harmfulness probes on grouped XSTest data and evaluates the frozen probes on JailbreakBench. Each section is implemented and validated before the next section is added.

## Section A: Load and verify data

This section loads the eight prepared activation datasets and checks their shapes, labels, ID alignment, numerical validity, and layer-0 control. Layer 0 is expected to be constant because every activation was extracted at the same assistant-boundary newline token. It will be retained only as a negative control.

In [ ]:
from __future__ import annotations

import csv
from pathlib import Path

import torch


ROOT = Path.cwd()
if not (ROOT / "data").is_dir():
    ROOT = ROOT.parent

PREPARED_DIR = ROOT / "analysis" / "prepared"
DATA_FILES = {"xstest": "xstest.csv", "jbb": "jbb.csv"}
MODELS = (
    "Qwen2.5-0.5B-Instruct",
    "Qwen2.5-1.5B-Instruct",
    "SmolLM2-360M-Instruct",
    "SmolLM2-1.7B-Instruct",
)
LABEL_MAP = {"safe": 0, "unsafe": 1}
EXPECTED_SHAPES = {
    ("xstest", "Qwen2.5-0.5B-Instruct"): (450, 25, 896),
    ("xstest", "Qwen2.5-1.5B-Instruct"): (450, 29, 1536),
    ("xstest", "SmolLM2-360M-Instruct"): (450, 33, 960),
    ("xstest", "SmolLM2-1.7B-Instruct"): (450, 25, 2048),
    ("jbb", "Qwen2.5-0.5B-Instruct"): (200, 25, 896),
    ("jbb", "Qwen2.5-1.5B-Instruct"): (200, 29, 1536),
    ("jbb", "SmolLM2-360M-Instruct"): (200, 33, 960),
    ("jbb", "SmolLM2-1.7B-Instruct"): (200, 25, 2048),
}

print(f"Repository root: {ROOT}")
print(f"Prepared data: {PREPARED_DIR}")

In [ ]:
def load_metadata(dataset: str) -> list[dict[str, str]]:
    path = ROOT / "data" / DATA_FILES[dataset]
    with path.open("r", encoding="utf-8-sig", newline="") as file:
        return list(csv.DictReader(file))


prepared_data: dict[tuple[str, str], dict] = {}
metadata_rows = {dataset: load_metadata(dataset) for dataset in DATA_FILES}

for dataset in DATA_FILES:
    for model in MODELS:
        path = PREPARED_DIR / f"{dataset}_{model}.pt"
        if not path.is_file():
            raise FileNotFoundError(f"Prepared probe dataset not found: {path}")
        prepared_data[(dataset, model)] = torch.load(
            path, map_location="cpu", weights_only=False
        )
        print(f"Loaded {path.name}")

In [ ]:
def verify_prepared_dataset(dataset: str, model: str, probe_data: dict) -> dict:
    required_keys = {
        "ids", "X", "y", "label_map",
        "num_examples", "activation_shape",
    }
    missing_keys = required_keys - set(probe_data)
    assert not missing_keys, f"Missing keys: {sorted(missing_keys)}"

    ids = [str(prompt_id) for prompt_id in probe_data["ids"]]
    X = probe_data["X"]
    y = probe_data["y"]
    rows = metadata_rows[dataset]

    expected_shape = EXPECTED_SHAPES[(dataset, model)]
    assert tuple(X.shape) == expected_shape, (
        f"{dataset}/{model}: expected X shape {expected_shape}, got {tuple(X.shape)}"
    )
    assert tuple(y.shape) == (expected_shape[0],)
    assert probe_data["num_examples"] == expected_shape[0]
    assert tuple(probe_data["activation_shape"]) == expected_shape[1:]
    assert probe_data["label_map"] == LABEL_MAP
    assert set(y.tolist()) == {0, 1}

    csv_ids = [str(row["id"]) for row in rows]
    expected_y = torch.tensor(
        [LABEL_MAP[str(row["label"]).strip().lower()] for row in rows],
        dtype=torch.long,
    )
    assert ids == csv_ids, f"{dataset}/{model}: prepared IDs do not match CSV order"
    assert torch.equal(y, expected_y), f"{dataset}/{model}: labels do not match CSV"
    assert torch.isfinite(X).all().item(), f"{dataset}/{model}: non-finite activation found"

    layer_zero_constant = torch.equal(
        X[:, 0, :], X[0, 0, :].expand_as(X[:, 0, :])
    )
    assert layer_zero_constant, f"{dataset}/{model}: layer 0 is not constant"

    safe_count = int((y == LABEL_MAP["safe"]).sum().item())
    unsafe_count = int((y == LABEL_MAP["unsafe"]).sum().item())
    return {
        "dataset": dataset,
        "model": model,
        "shape": tuple(X.shape),
        "dtype": str(X.dtype),
        "safe": safe_count,
        "unsafe": unsafe_count,
        "ids_match": ids == csv_ids,
        "all_finite": bool(torch.isfinite(X).all().item()),
        "layer_0_constant": layer_zero_constant,
    }

In [ ]:
verification_results = [
    verify_prepared_dataset(dataset, model, probe_data)
    for (dataset, model), probe_data in prepared_data.items()
]

for result in verification_results:
    print(
        f"{result['dataset']:6} | {result['model']:26} | "
        f"shape={result['shape']} | {result['dtype']} | "
        f"safe={result['safe']} unsafe={result['unsafe']} | "
        f"IDs={result['ids_match']} finite={result['all_finite']} "
        f"layer0_constant={result['layer_0_constant']}"
    )

print("\nSection A passed for every prepared dataset.")

## Section B: Define grouped XSTest folds

The legacy random row split is replaced with nested stratified group folds. Non-empty `focus` values define semantic groups. Each row with an empty `focus` receives its own deterministic fallback group based on its prompt ID. The five outer folds estimate generalization; the three inner folds will be used only for selecting layer and regularization strength. JailbreakBench is not used here.

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold


RANDOM_SEED = 42
OUTER_SPLITS = 5
INNER_SPLITS = 3

xstest_rows = metadata_rows["xstest"]
xstest_ids = np.asarray([str(row["id"]) for row in xstest_rows])
xstest_y = prepared_data[("xstest", MODELS[0])]["y"].numpy()
xstest_groups = np.asarray([
    f"focus:{row['focus'].strip()}"
    if row["focus"].strip()
    else f"empty_focus:{row['id']}"
    for row in xstest_rows
])

assert len(xstest_ids) == len(xstest_y) == len(xstest_groups) == 450
assert xstest_ids.tolist() == prepared_data[("xstest", MODELS[0])]["ids"]

empty_focus_count = sum(not row["focus"].strip() for row in xstest_rows)
print(f"XSTest examples: {len(xstest_ids)}")
print(f"Unique groups: {len(np.unique(xstest_groups))}")
print(f"Empty-focus fallback groups: {empty_focus_count}")

In [ ]:
def binary_label_counts(labels: np.ndarray) -> tuple[int, int]:
    return int((labels == 0).sum()), int((labels == 1).sum())


outer_splitter = StratifiedGroupKFold(
    n_splits=OUTER_SPLITS,
    shuffle=True,
    random_state=RANDOM_SEED,
)

outer_folds: list[dict] = []
outer_test_coverage = np.zeros(len(xstest_y), dtype=np.int64)

for outer_fold, (train_indices, test_indices) in enumerate(
    outer_splitter.split(
        X=np.zeros((len(xstest_y), 1)),
        y=xstest_y,
        groups=xstest_groups,
    )
):
    train_groups = set(xstest_groups[train_indices])
    test_groups = set(xstest_groups[test_indices])
    assert train_groups.isdisjoint(test_groups)
    assert set(xstest_y[train_indices]) == {0, 1}
    assert set(xstest_y[test_indices]) == {0, 1}
    outer_test_coverage[test_indices] += 1

    inner_splitter = StratifiedGroupKFold(
        n_splits=INNER_SPLITS,
        shuffle=True,
        random_state=RANDOM_SEED + outer_fold + 1,
    )
    inner_folds: list[dict] = []

    for inner_fold, (inner_train_positions, validation_positions) in enumerate(
        inner_splitter.split(
            X=np.zeros((len(train_indices), 1)),
            y=xstest_y[train_indices],
            groups=xstest_groups[train_indices],
        )
    ):
        inner_train_indices = train_indices[inner_train_positions]
        validation_indices = train_indices[validation_positions]
        inner_train_groups = set(xstest_groups[inner_train_indices])
        validation_groups = set(xstest_groups[validation_indices])

        assert inner_train_groups.isdisjoint(validation_groups)
        assert set(xstest_y[inner_train_indices]) == {0, 1}
        assert set(xstest_y[validation_indices]) == {0, 1}
        assert set(inner_train_indices).isdisjoint(test_indices)
        assert set(validation_indices).isdisjoint(test_indices)

        inner_folds.append({
            "inner_fold": inner_fold,
            "train_indices": inner_train_indices,
            "validation_indices": validation_indices,
        })

    outer_folds.append({
        "outer_fold": outer_fold,
        "train_indices": train_indices,
        "test_indices": test_indices,
        "inner_folds": inner_folds,
    })

assert np.all(outer_test_coverage == 1), (
    "Every XSTest example must occur in exactly one outer test fold"
)

In [ ]:
for fold in outer_folds:
    train_indices = fold["train_indices"]
    test_indices = fold["test_indices"]
    train_safe, train_unsafe = binary_label_counts(xstest_y[train_indices])
    test_safe, test_unsafe = binary_label_counts(xstest_y[test_indices])
    print(
        f"Outer {fold['outer_fold']}: "
        f"train={len(train_indices)} (safe={train_safe}, unsafe={train_unsafe}), "
        f"test={len(test_indices)} (safe={test_safe}, unsafe={test_unsafe}), "
        f"train_groups={len(set(xstest_groups[train_indices]))}, "
        f"test_groups={len(set(xstest_groups[test_indices]))}"
    )
    for inner in fold["inner_folds"]:
        inner_train = inner["train_indices"]
        validation = inner["validation_indices"]
        inner_safe, inner_unsafe = binary_label_counts(xstest_y[inner_train])
        val_safe, val_unsafe = binary_label_counts(xstest_y[validation])
        print(
            f"  Inner {inner['inner_fold']}: "
            f"train={len(inner_train)} ({inner_safe}/{inner_unsafe}), "
            f"validation={len(validation)} ({val_safe}/{val_unsafe})"
        )

print("\nSection B passed: grouped nested folds are disjoint, stratified, and exhaustive.")

## Section C: Train layer-wise probes

For each model and outer fold, the inner folds evaluate every layer and regularization value. Each scaler is fitted only on the relevant training split. Mean inner-fold AUROC selects the layer and `C`; exact ties prefer stronger regularization and then the earlier layer. Layer 0 is evaluated as a control but excluded from selection. The selected configuration is then fitted on the complete outer-training split and evaluated once on the untouched outer-test split.

In [ ]:
from collections import defaultdict

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler


C_GRID = (0.01, 0.1, 1.0, 10.0, 100.0)
MAX_ITER = 20000


def calculate_probe_metrics(
    true_labels: np.ndarray,
    margins: np.ndarray,
    predictions: np.ndarray,
) -> dict[str, float]:
    return {
        "auroc": float(roc_auc_score(true_labels, margins)),
        "auprc": float(average_precision_score(true_labels, margins)),
        "macro_f1": float(f1_score(true_labels, predictions, average="macro")),
        "balanced_accuracy": float(
            balanced_accuracy_score(true_labels, predictions)
        ),
    }


def fit_logistic_probe(
    train_features: np.ndarray,
    train_labels: np.ndarray,
    evaluation_features: np.ndarray,
    regularization_c: float,
    random_state: int = RANDOM_SEED,
) -> tuple[StandardScaler, LogisticRegression, np.ndarray, np.ndarray]:
    scaler = StandardScaler()
    scaled_train = scaler.fit_transform(train_features)
    scaled_evaluation = scaler.transform(evaluation_features)
    probe = LogisticRegression(
        C=regularization_c,
        solver="liblinear",
        dual=True,
        max_iter=MAX_ITER,
        random_state=random_state,
    )
    probe.fit(scaled_train, train_labels)
    margins = probe.decision_function(scaled_evaluation)
    predictions = (margins >= 0.0).astype(np.int64)
    return scaler, probe, margins, predictions


def write_csv_rows(path: Path, rows: list[dict], fieldnames: list[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

In [ ]:
def evaluate_inner_grid(
    model: str,
    activations: np.ndarray,
    labels: np.ndarray,
    outer_fold: dict,
) -> list[dict]:
    records: list[dict] = []

    for layer in range(activations.shape[1]):
        for inner in outer_fold["inner_folds"]:
            train_indices = inner["train_indices"]
            validation_indices = inner["validation_indices"]

            for regularization_c in C_GRID:
                _, probe, margins, predictions = fit_logistic_probe(
                    train_features=activations[train_indices, layer, :],
                    train_labels=labels[train_indices],
                    evaluation_features=activations[validation_indices, layer, :],
                    regularization_c=regularization_c,
                )
                metrics = calculate_probe_metrics(
                    labels[validation_indices], margins, predictions
                )
                records.append({
                    "model": model,
                    "outer_fold": outer_fold["outer_fold"],
                    "inner_fold": inner["inner_fold"],
                    "layer": layer,
                    "C": regularization_c,
                    **metrics,
                    "n_iter": int(probe.n_iter_[0]),
                })

    return records


def summarize_inner_grid(records: list[dict]) -> list[dict]:
    grouped: dict[tuple[int, float], list[dict]] = defaultdict(list)
    for record in records:
        grouped[(record["layer"], record["C"])].append(record)

    summaries: list[dict] = []
    for (layer, regularization_c), fold_records in grouped.items():
        summaries.append({
            "layer": layer,
            "C": regularization_c,
            "mean_auroc": float(np.mean([row["auroc"] for row in fold_records])),
            "mean_auprc": float(np.mean([row["auprc"] for row in fold_records])),
            "mean_macro_f1": float(
                np.mean([row["macro_f1"] for row in fold_records])
            ),
            "mean_balanced_accuracy": float(
                np.mean([row["balanced_accuracy"] for row in fold_records])
            ),
        })
    return summaries


def selection_key(summary: dict) -> tuple[float, float, int]:
    return (-summary["mean_auroc"], summary["C"], summary["layer"])

In [ ]:
inner_grid_records: list[dict] = []
layerwise_outer_records: list[dict] = []
selected_outer_records: list[dict] = []
xstest_oof_predictions: list[dict] = []

for model in MODELS:
    activations = prepared_data[("xstest", model)]["X"].numpy()
    labels = prepared_data[("xstest", model)]["y"].numpy()
    model_oof_count = 0

    for outer_fold in outer_folds:
        fold_grid_records = evaluate_inner_grid(
            model=model,
            activations=activations,
            labels=labels,
            outer_fold=outer_fold,
        )
        inner_grid_records.extend(fold_grid_records)
        inner_summaries = summarize_inner_grid(fold_grid_records)

        selectable_summaries = [row for row in inner_summaries if row["layer"] > 0]
        selected_summary = min(selectable_summaries, key=selection_key)
        selected_layer = selected_summary["layer"]
        selected_c = selected_summary["C"]
        train_indices = outer_fold["train_indices"]
        test_indices = outer_fold["test_indices"]
        selected_margins: np.ndarray | None = None
        selected_predictions: np.ndarray | None = None
        selected_metrics: dict[str, float] | None = None
        selected_n_iter: int | None = None

        for layer in range(activations.shape[1]):
            best_for_layer = min(
                (row for row in inner_summaries if row["layer"] == layer),
                key=selection_key,
            )
            _, probe, margins, predictions = fit_logistic_probe(
                train_features=activations[train_indices, layer, :],
                train_labels=labels[train_indices],
                evaluation_features=activations[test_indices, layer, :],
                regularization_c=best_for_layer["C"],
            )
            outer_metrics = calculate_probe_metrics(
                labels[test_indices], margins, predictions
            )
            layerwise_outer_records.append({
                "model": model,
                "outer_fold": outer_fold["outer_fold"],
                "layer": layer,
                "C": best_for_layer["C"],
                "inner_mean_auroc": best_for_layer["mean_auroc"],
                **outer_metrics,
                "n_iter": int(probe.n_iter_[0]),
            })

            if layer == selected_layer:
                assert best_for_layer["C"] == selected_c
                selected_margins = margins
                selected_predictions = predictions
                selected_metrics = outer_metrics
                selected_n_iter = int(probe.n_iter_[0])

        assert selected_margins is not None
        assert selected_predictions is not None
        assert selected_metrics is not None
        assert selected_n_iter is not None
        selected_outer_records.append({
            "model": model,
            "outer_fold": outer_fold["outer_fold"],
            "layer": selected_layer,
            "C": selected_c,
            "inner_mean_auroc": selected_summary["mean_auroc"],
            **selected_metrics,
            "n_iter": selected_n_iter,
        })

        for position, prompt_index in enumerate(test_indices):
            xstest_oof_predictions.append({
                "model": model,
                "layer": selected_layer,
                "outer_fold": outer_fold["outer_fold"],
                "prompt_id": xstest_ids[prompt_index],
                "group": xstest_groups[prompt_index],
                "true_label": int(labels[prompt_index]),
                "probe_margin": float(selected_margins[position]),
                "prediction": int(selected_predictions[position]),
            })
        model_oof_count += len(test_indices)

    assert model_oof_count == 450
    print(f"Completed nested probe evaluation for {model}")

In [ ]:
assert len(xstest_oof_predictions) == len(MODELS) * 450
assert len({(row["model"], row["prompt_id"]) for row in xstest_oof_predictions}) == len(MODELS) * 450
assert all(np.isfinite(row["probe_margin"]) for row in xstest_oof_predictions)
assert all(row["n_iter"] < MAX_ITER for row in inner_grid_records)
assert all(row["n_iter"] < MAX_ITER for row in layerwise_outer_records)

xstest_oof_predictions.sort(key=lambda row: (row["model"], row["prompt_id"]))
xstest_oof_summary: list[dict] = []
for model in MODELS:
    model_rows = [row for row in xstest_oof_predictions if row["model"] == model]
    true_labels = np.asarray([row["true_label"] for row in model_rows])
    margins = np.asarray([row["probe_margin"] for row in model_rows])
    predictions = np.asarray([row["prediction"] for row in model_rows])
    metrics = calculate_probe_metrics(true_labels, margins, predictions)
    xstest_oof_summary.append({"model": model, **metrics})
    selected = [row for row in selected_outer_records if row["model"] == model]
    selections = [(row["layer"], row["C"]) for row in selected]
    print(
        f"{model}: AUROC={metrics['auroc']:.3f}, AUPRC={metrics['auprc']:.3f}, "
        f"macro-F1={metrics['macro_f1']:.3f}, "
        f"balanced accuracy={metrics['balanced_accuracy']:.3f}"
    )
    print(f"  selected (layer, C) by outer fold: {selections}")

write_csv_rows(
    ROOT / "analysis" / "predictions" / "xstest_oof_predictions.csv",
    xstest_oof_predictions,
    ["model", "layer", "outer_fold", "prompt_id", "group",
     "true_label", "probe_margin", "prediction"],
)
write_csv_rows(
    ROOT / "analysis" / "metrics" / "xstest_layerwise_outer.csv",
    layerwise_outer_records,
    ["model", "outer_fold", "layer", "C", "inner_mean_auroc",
     "auroc", "auprc", "macro_f1", "balanced_accuracy", "n_iter"],
)
write_csv_rows(
    ROOT / "analysis" / "metrics" / "xstest_selected_outer.csv",
    selected_outer_records,
    ["model", "outer_fold", "layer", "C", "inner_mean_auroc",
     "auroc", "auprc", "macro_f1", "balanced_accuracy", "n_iter"],
)
print(
    f"\nSection C passed: all {len(xstest_oof_predictions)} "
    "selected-probe OOF predictions are finite and unique."
)


## Section D: Baselines and controls

The surface baselines use the same nested grouped folds and select `C` by mean inner AUROC. Word TF-IDF uses word unigrams and bigrams; character TF-IDF uses within-word 3–5 character n-grams. Length uses character and whitespace-word counts. Punctuation uses counts of `? ! . , : ; - ' \" ( )`. First-token uses a one-hot representation of the first lowercase lexical token. Every vectorizer or scaler is fitted on the relevant training fold only.

Layer 0 is reconstructed as a model-specific constant-embedding control. The shuffled-label control keeps each outer fold's selected activation layer and `C`, permutes only its training labels, and repeats this with three fixed seeds before evaluation against the untouched true test labels.

In [ ]:
import re

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer


BASELINES = ("word_tfidf", "character_tfidf", "length", "punctuation", "first_token")
PUNCTUATION_MARKS = tuple("?!.,:;-\'\"()")
FIRST_TOKEN_PATTERN = re.compile(r"\b\w+\b", flags=re.UNICODE)
SHUFFLE_SEEDS = (101, 202, 303)
xstest_prompts = np.asarray([str(row["prompt"]) for row in xstest_rows])


def prompt_length_features(prompts: np.ndarray) -> np.ndarray:
    return np.asarray([
        [len(prompt), len(prompt.split())]
        for prompt in prompts
    ], dtype=np.float64)


def punctuation_features(prompts: np.ndarray) -> np.ndarray:
    return np.asarray([
        [prompt.count(mark) for mark in PUNCTUATION_MARKS]
        for prompt in prompts
    ], dtype=np.float64)


def first_token_text(prompt: str) -> str:
    match = FIRST_TOKEN_PATTERN.search(prompt.lower())
    return match.group(0) if match else "<empty>"


def build_baseline_features(
    baseline: str,
    training_prompts: np.ndarray,
    evaluation_prompts: np.ndarray,
):
    if baseline == "word_tfidf":
        transformer = TfidfVectorizer(
            lowercase=True, ngram_range=(1, 2), min_df=2, sublinear_tf=True
        )
        return (
            transformer.fit_transform(training_prompts),
            transformer.transform(evaluation_prompts),
        )
    if baseline == "character_tfidf":
        transformer = TfidfVectorizer(
            analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True
        )
        return (
            transformer.fit_transform(training_prompts),
            transformer.transform(evaluation_prompts),
        )
    if baseline == "length":
        scaler = StandardScaler()
        return (
            scaler.fit_transform(prompt_length_features(training_prompts)),
            scaler.transform(prompt_length_features(evaluation_prompts)),
        )
    if baseline == "punctuation":
        scaler = StandardScaler()
        return (
            scaler.fit_transform(punctuation_features(training_prompts)),
            scaler.transform(punctuation_features(evaluation_prompts)),
        )
    if baseline == "first_token":
        transformer = CountVectorizer(binary=True, token_pattern=r"(?u)\b\w+\b")
        train_tokens = [first_token_text(prompt) for prompt in training_prompts]
        evaluation_tokens = [first_token_text(prompt) for prompt in evaluation_prompts]
        return (
            transformer.fit_transform(train_tokens),
            transformer.transform(evaluation_tokens),
        )
    raise ValueError(f"Unknown baseline: {baseline}")


def fit_surface_probe(
    train_features,
    train_labels: np.ndarray,
    evaluation_features,
    regularization_c: float,
) -> tuple[LogisticRegression, np.ndarray, np.ndarray]:
    probe = LogisticRegression(
        C=regularization_c,
        solver="liblinear",
        dual=train_features.shape[1] > train_features.shape[0],
        max_iter=MAX_ITER,
        random_state=RANDOM_SEED,
    )
    probe.fit(train_features, train_labels)
    margins = probe.decision_function(evaluation_features)
    predictions = (margins >= 0.0).astype(np.int64)
    return probe, margins, predictions

In [ ]:
baseline_inner_records: list[dict] = []
baseline_outer_records: list[dict] = []
baseline_oof_predictions: list[dict] = []

for baseline in BASELINES:
    for outer_fold in outer_folds:
        fold_inner_records: list[dict] = []
        for inner in outer_fold["inner_folds"]:
            train_indices = inner["train_indices"]
            validation_indices = inner["validation_indices"]
            train_features, validation_features = build_baseline_features(
                baseline,
                xstest_prompts[train_indices],
                xstest_prompts[validation_indices],
            )
            for regularization_c in C_GRID:
                probe, margins, predictions = fit_surface_probe(
                    train_features,
                    xstest_y[train_indices],
                    validation_features,
                    regularization_c,
                )
                metrics = calculate_probe_metrics(
                    xstest_y[validation_indices], margins, predictions
                )
                record = {
                    "baseline": baseline,
                    "outer_fold": outer_fold["outer_fold"],
                    "inner_fold": inner["inner_fold"],
                    "C": regularization_c,
                    **metrics,
                    "n_iter": int(probe.n_iter_[0]),
                }
                fold_inner_records.append(record)
                baseline_inner_records.append(record)

        c_scores: dict[float, list[float]] = defaultdict(list)
        for record in fold_inner_records:
            c_scores[record["C"]].append(record["auroc"])
        selected_c = min(
            C_GRID,
            key=lambda regularization_c: (
                -float(np.mean(c_scores[regularization_c])), regularization_c
            ),
        )

        train_indices = outer_fold["train_indices"]
        test_indices = outer_fold["test_indices"]
        train_features, test_features = build_baseline_features(
            baseline, xstest_prompts[train_indices], xstest_prompts[test_indices]
        )
        probe, margins, predictions = fit_surface_probe(
            train_features, xstest_y[train_indices], test_features, selected_c
        )
        metrics = calculate_probe_metrics(xstest_y[test_indices], margins, predictions)
        baseline_outer_records.append({
            "baseline": baseline,
            "outer_fold": outer_fold["outer_fold"],
            "C": selected_c,
            "inner_mean_auroc": float(np.mean(c_scores[selected_c])),
            **metrics,
            "n_iter": int(probe.n_iter_[0]),
        })
        for position, prompt_index in enumerate(test_indices):
            baseline_oof_predictions.append({
                "baseline": baseline,
                "outer_fold": outer_fold["outer_fold"],
                "prompt_id": xstest_ids[prompt_index],
                "group": xstest_groups[prompt_index],
                "true_label": int(xstest_y[prompt_index]),
                "probe_margin": float(margins[position]),
                "prediction": int(predictions[position]),
            })
    print(f"Completed grouped nested evaluation for {baseline}")

In [ ]:
control_oof_predictions: list[dict] = []
control_summary: list[dict] = []

for model in MODELS:
    activations = prepared_data[("xstest", model)]["X"].numpy()
    labels = prepared_data[("xstest", model)]["y"].numpy()

    layer_zero_rows: list[dict] = []
    for outer_fold in outer_folds:
        matching_record = next(
            row for row in layerwise_outer_records
            if row["model"] == model
            and row["outer_fold"] == outer_fold["outer_fold"]
            and row["layer"] == 0
        )
        train_indices = outer_fold["train_indices"]
        test_indices = outer_fold["test_indices"]
        _, probe, margins, predictions = fit_logistic_probe(
            activations[train_indices, 0, :],
            labels[train_indices],
            activations[test_indices, 0, :],
            matching_record["C"],
        )
        for position, prompt_index in enumerate(test_indices):
            row = {
                "control": "layer_0",
                "model": model,
                "seed": "",
                "outer_fold": outer_fold["outer_fold"],
                "prompt_id": xstest_ids[prompt_index],
                "true_label": int(labels[prompt_index]),
                "probe_margin": float(margins[position]),
                "prediction": int(predictions[position]),
            }
            layer_zero_rows.append(row)
            control_oof_predictions.append(row)

    layer_zero_metrics = calculate_probe_metrics(
        np.asarray([row["true_label"] for row in layer_zero_rows]),
        np.asarray([row["probe_margin"] for row in layer_zero_rows]),
        np.asarray([row["prediction"] for row in layer_zero_rows]),
    )
    control_summary.append({
        "method": "layer_0", "model": model, "seed": "",
        **layer_zero_metrics,
    })

    selected_by_fold = {
        row["outer_fold"]: row
        for row in selected_outer_records if row["model"] == model
    }
    for shuffle_seed in SHUFFLE_SEEDS:
        shuffled_rows: list[dict] = []
        for outer_fold in outer_folds:
            selected = selected_by_fold[outer_fold["outer_fold"]]
            train_indices = outer_fold["train_indices"]
            test_indices = outer_fold["test_indices"]
            shuffled_labels = labels[train_indices].copy()
            rng = np.random.default_rng(shuffle_seed + outer_fold["outer_fold"])
            rng.shuffle(shuffled_labels)
            assert not np.array_equal(shuffled_labels, labels[train_indices])
            _, probe, margins, predictions = fit_logistic_probe(
                activations[train_indices, selected["layer"], :],
                shuffled_labels,
                activations[test_indices, selected["layer"], :],
                selected["C"],
                random_state=shuffle_seed,
            )
            for position, prompt_index in enumerate(test_indices):
                row = {
                    "control": "shuffled_labels",
                    "model": model,
                    "seed": shuffle_seed,
                    "outer_fold": outer_fold["outer_fold"],
                    "prompt_id": xstest_ids[prompt_index],
                    "true_label": int(labels[prompt_index]),
                    "probe_margin": float(margins[position]),
                    "prediction": int(predictions[position]),
                }
                shuffled_rows.append(row)
                control_oof_predictions.append(row)

        shuffled_metrics = calculate_probe_metrics(
            np.asarray([row["true_label"] for row in shuffled_rows]),
            np.asarray([row["probe_margin"] for row in shuffled_rows]),
            np.asarray([row["prediction"] for row in shuffled_rows]),
        )
        control_summary.append({
            "method": "shuffled_labels", "model": model,
            "seed": shuffle_seed, **shuffled_metrics,
        })

In [ ]:
assert len(baseline_inner_records) == len(BASELINES) * OUTER_SPLITS * INNER_SPLITS * len(C_GRID)
assert len(baseline_oof_predictions) == len(BASELINES) * 450
assert len({(row["baseline"], row["prompt_id"]) for row in baseline_oof_predictions}) == len(BASELINES) * 450
assert all(row["n_iter"] < MAX_ITER for row in baseline_inner_records)
assert all(row["n_iter"] < MAX_ITER for row in baseline_outer_records)
assert all(np.isfinite(row["probe_margin"]) for row in baseline_oof_predictions)
assert all(np.isfinite(row["probe_margin"]) for row in control_oof_predictions)

baseline_summary: list[dict] = []
for baseline in BASELINES:
    rows = [row for row in baseline_oof_predictions if row["baseline"] == baseline]
    metrics = calculate_probe_metrics(
        np.asarray([row["true_label"] for row in rows]),
        np.asarray([row["probe_margin"] for row in rows]),
        np.asarray([row["prediction"] for row in rows]),
    )
    baseline_summary.append({
        "method": baseline, "model": "shared_text_baseline",
        "seed": "", **metrics,
    })

activation_summary = [
    {"method": "selected_activation_probe", "model": row["model"],
     "seed": "", "auroc": row["auroc"], "auprc": row["auprc"],
     "macro_f1": row["macro_f1"],
     "balanced_accuracy": row["balanced_accuracy"]}
    for row in xstest_oof_summary
]
comparison_summary = activation_summary + baseline_summary + control_summary

for row in comparison_summary:
    seed_text = f" seed={row['seed']}" if row["seed"] != "" else ""
    print(
        f"{row['method']:25} | {row['model']:26}{seed_text:10} | "
        f"AUROC={row['auroc']:.3f} AUPRC={row['auprc']:.3f} "
        f"macro-F1={row['macro_f1']:.3f} BA={row['balanced_accuracy']:.3f}"
    )

write_csv_rows(
    ROOT / "analysis" / "metrics" / "xstest_baseline_comparison.csv",
    comparison_summary,
    ["method", "model", "seed", "auroc", "auprc",
     "macro_f1", "balanced_accuracy"],
)

print("\nSection D passed: every baseline and control produced complete finite OOF predictions.")


## Section E: Select and freeze the final probes

Final layer and `C` selection uses only XSTest. The same five grouped folds evaluate every non-control layer and `C` across all XSTest examples; these scores select the model that will be fitted on all 450 prompts. This selection pass is for the externally evaluated final model and is not reported as a new unbiased XSTest estimate—the nested OOF estimate from Section C remains the XSTest result.

Each frozen checkpoint contains only the selected layer, scaler parameters, logistic-regression parameters, labels, training IDs, and XSTest selection metadata. A SHA-256 digest is recorded immediately after saving. Section F must load these files without fitting or tuning anything on JailbreakBench.

In [ ]:
import hashlib


final_selection_fold_records: list[dict] = []

for model in MODELS:
    activations = prepared_data[("xstest", model)]["X"].numpy()
    labels = prepared_data[("xstest", model)]["y"].numpy()

    for outer_fold in outer_folds:
        train_indices = outer_fold["train_indices"]
        test_indices = outer_fold["test_indices"]
        for layer in range(1, activations.shape[1]):
            scaler = StandardScaler()
            scaled_train = scaler.fit_transform(activations[train_indices, layer, :])
            scaled_test = scaler.transform(activations[test_indices, layer, :])

            for regularization_c in C_GRID:
                probe = LogisticRegression(
                    C=regularization_c,
                    solver="liblinear",
                    dual=True,
                    max_iter=MAX_ITER,
                    random_state=RANDOM_SEED,
                )
                probe.fit(scaled_train, labels[train_indices])
                margins = probe.decision_function(scaled_test)
                predictions = (margins >= 0.0).astype(np.int64)
                metrics = calculate_probe_metrics(
                    labels[test_indices], margins, predictions
                )
                final_selection_fold_records.append({
                    "model": model,
                    "fold": outer_fold["outer_fold"],
                    "layer": layer,
                    "C": regularization_c,
                    **metrics,
                    "n_iter": int(probe.n_iter_[0]),
                })
    print(f"Completed final XSTest-only selection grid for {model}")

In [ ]:
selection_groups: dict[tuple[str, int, float], list[dict]] = defaultdict(list)
for record in final_selection_fold_records:
    selection_groups[(record["model"], record["layer"], record["C"])].append(record)

final_selection_grid: list[dict] = []
for (model, layer, regularization_c), records in selection_groups.items():
    final_selection_grid.append({
        "model": model,
        "layer": layer,
        "C": regularization_c,
        "mean_auroc": float(np.mean([row["auroc"] for row in records])),
        "std_auroc": float(np.std([row["auroc"] for row in records], ddof=1)),
        "mean_auprc": float(np.mean([row["auprc"] for row in records])),
        "mean_macro_f1": float(np.mean([row["macro_f1"] for row in records])),
        "mean_balanced_accuracy": float(
            np.mean([row["balanced_accuracy"] for row in records])
        ),
    })

assert all(row["n_iter"] < MAX_ITER for row in final_selection_fold_records)
assert len(final_selection_grid) == sum(
    (prepared_data[("xstest", model)]["X"].shape[1] - 1) * len(C_GRID)
    for model in MODELS
)

In [ ]:
def margins_from_frozen_checkpoint(
    checkpoint: dict, activations: np.ndarray
) -> np.ndarray:
    layer = int(checkpoint["layer"])
    features = activations[:, layer, :].astype(np.float32, copy=True)
    mean = checkpoint["scaler_mean"].numpy().astype(np.float32)
    scale = checkpoint["scaler_scale"].numpy().astype(np.float32)
    coefficient = checkpoint["coefficient"].numpy().reshape(-1)
    intercept = float(checkpoint["intercept"].item())
    features -= mean
    features /= scale
    return features @ coefficient + intercept


checkpoint_dir = ROOT / "analysis" / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
frozen_probe_paths: dict[str, Path] = {}
frozen_probe_hashes: dict[str, str] = {}
frozen_probe_selection: list[dict] = []

for model in MODELS:
    candidates = [row for row in final_selection_grid if row["model"] == model]
    selected = min(
        candidates,
        key=lambda row: (-row["mean_auroc"], row["C"], row["layer"]),
    )
    layer = selected["layer"]
    regularization_c = selected["C"]
    activations = prepared_data[("xstest", model)]["X"].numpy()
    labels = prepared_data[("xstest", model)]["y"].numpy()

    scaler = StandardScaler()
    scaled_activations = scaler.fit_transform(activations[:, layer, :])
    probe = LogisticRegression(
        C=regularization_c,
        solver="liblinear",
        dual=True,
        max_iter=MAX_ITER,
        random_state=RANDOM_SEED,
    )
    probe.fit(scaled_activations, labels)
    assert int(probe.n_iter_[0]) < MAX_ITER

    checkpoint = {
        "format_version": 1,
        "model": model,
        "training_dataset": "xstest",
        "training_ids": xstest_ids.tolist(),
        "label_map": LABEL_MAP,
        "layer": int(layer),
        "C": float(regularization_c),
        "selection_metric": "mean_grouped_5_fold_xstest_auroc",
        "selection_score": float(selected["mean_auroc"]),
        "scaler_mean": torch.tensor(scaler.mean_, dtype=torch.float64),
        "scaler_scale": torch.tensor(scaler.scale_, dtype=torch.float64),
        "coefficient": torch.tensor(probe.coef_, dtype=torch.float64),
        "intercept": torch.tensor(probe.intercept_, dtype=torch.float64),
        "classes": torch.tensor(probe.classes_, dtype=torch.long),
    }
    checkpoint_path = checkpoint_dir / f"xstest_{model}_probe.pt"
    torch.save(checkpoint, checkpoint_path)

    reloaded = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    saved_margins = margins_from_frozen_checkpoint(reloaded, activations)
    sklearn_margins = probe.decision_function(scaled_activations)
    assert np.allclose(saved_margins, sklearn_margins, rtol=1e-10, atol=1e-10)
    assert reloaded["training_ids"] == xstest_ids.tolist()

    checkpoint_hash = hashlib.sha256(checkpoint_path.read_bytes()).hexdigest()
    frozen_probe_paths[model] = checkpoint_path
    frozen_probe_hashes[model] = checkpoint_hash
    frozen_probe_selection.append({
        "model": model,
        "layer": layer,
        "C": regularization_c,
        "xstest_cv_auroc": selected["mean_auroc"],
        "checkpoint_path": checkpoint_path.relative_to(ROOT).as_posix(),
        "sha256": checkpoint_hash,
    })
    print(
        f"Frozen {model}: layer={layer}, C={regularization_c}, "
        f"XSTest selection AUROC={selected['mean_auroc']:.3f}, "
        f"SHA-256={checkpoint_hash[:12]}..."
    )

In [ ]:
write_csv_rows(
    ROOT / "analysis" / "metrics" / "frozen_probe_selection.csv",
    frozen_probe_selection,
    ["model", "layer", "C", "xstest_cv_auroc",
     "checkpoint_path", "sha256"],
)

assert set(frozen_probe_paths) == set(MODELS)
print("\nSection E passed: every XSTest-only probe was saved, reloaded, and frozen.")


## Section F: Evaluate the frozen probes on JailbreakBench

This is the untouched external evaluation. Each checkpoint is loaded from disk and its SHA-256 digest is checked before and after inference. No scaler, probe, layer, threshold, or regularization value is fitted or selected on JailbreakBench. Confidence intervals use a paired cluster bootstrap: each resample draws 100 JailbreakBench `index` values with replacement and includes both the benign and misuse prompt from every sampled pair.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix


BOOTSTRAP_SAMPLES = 10_000
CONFIDENCE_LEVEL = 0.95
jbb_rows = metadata_rows["jbb"]
jbb_ids = np.asarray([str(row["id"]) for row in jbb_rows])
jbb_labels = np.asarray([LABEL_MAP[str(row["label"]).strip().lower()] for row in jbb_rows])
jbb_pair_indices = np.asarray([int(row["index"]) for row in jbb_rows])
unique_jbb_pairs = np.unique(jbb_pair_indices)

assert len(jbb_rows) == 200
assert len(unique_jbb_pairs) == 100
assert set(unique_jbb_pairs.tolist()) == set(range(100))
for pair_index in unique_jbb_pairs:
    pair_labels = jbb_labels[jbb_pair_indices == pair_index]
    assert len(pair_labels) == 2
    assert set(pair_labels.tolist()) == {0, 1}
assert not ({row["prompt"] for row in jbb_rows} & {row["prompt"] for row in xstest_rows})

pair_positions = np.stack([
    np.flatnonzero(jbb_pair_indices == pair_index)
    for pair_index in unique_jbb_pairs
])
assert pair_positions.shape == (100, 2)
print("JBB pairing verified: 100 complete benign/misuse pairs and no exact XSTest overlap.")

In [ ]:
def paired_bootstrap_intervals(
    true_labels: np.ndarray,
    margins: np.ndarray,
    predictions: np.ndarray,
    seed: int,
) -> dict[str, tuple[float, float]]:
    rng = np.random.default_rng(seed)
    samples: dict[str, np.ndarray] = {
        metric: np.empty(BOOTSTRAP_SAMPLES, dtype=np.float64)
        for metric in ("auroc", "auprc", "macro_f1", "balanced_accuracy")
    }

    for sample_number in range(BOOTSTRAP_SAMPLES):
        sampled_pair_rows = rng.integers(0, len(pair_positions), size=len(pair_positions))
        sampled_positions = pair_positions[sampled_pair_rows].reshape(-1)
        metrics = calculate_probe_metrics(
            true_labels[sampled_positions],
            margins[sampled_positions],
            predictions[sampled_positions],
        )
        for metric, value in metrics.items():
            samples[metric][sample_number] = value

    alpha = 1.0 - CONFIDENCE_LEVEL
    return {
        metric: (
            float(np.quantile(values, alpha / 2.0)),
            float(np.quantile(values, 1.0 - alpha / 2.0)),
        )
        for metric, values in samples.items()
    }


def summarize_margin_distribution(values: np.ndarray) -> dict[str, float]:
    return {
        "count": int(len(values)),
        "mean": float(np.mean(values)),
        "std": float(np.std(values, ddof=1)),
        "minimum": float(np.min(values)),
        "q25": float(np.quantile(values, 0.25)),
        "median": float(np.median(values)),
        "q75": float(np.quantile(values, 0.75)),
        "maximum": float(np.max(values)),
        "probe_positive_rate": float(np.mean(values >= 0.0)),
    }

In [ ]:
jbb_predictions: list[dict] = []
jbb_metric_intervals: list[dict] = []
jbb_confusion_matrices: list[dict] = []
jbb_margin_summaries: list[dict] = []
jbb_results_by_model: dict[str, dict] = {}

for model_number, model in enumerate(MODELS):
    checkpoint_path = frozen_probe_paths[model]
    hash_before = hashlib.sha256(checkpoint_path.read_bytes()).hexdigest()
    assert hash_before == frozen_probe_hashes[model]
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    assert checkpoint["model"] == model
    assert checkpoint["training_dataset"] == "xstest"

    jbb_activations = prepared_data[("jbb", model)]["X"].numpy()
    assert jbb_ids.tolist() == prepared_data[("jbb", model)]["ids"]
    margins = margins_from_frozen_checkpoint(checkpoint, jbb_activations)
    predictions = (margins >= 0.0).astype(np.int64)
    assert np.isfinite(margins).all()

    point_metrics = calculate_probe_metrics(jbb_labels, margins, predictions)
    intervals = paired_bootstrap_intervals(
        jbb_labels, margins, predictions, seed=RANDOM_SEED + model_number
    )
    for metric, estimate in point_metrics.items():
        lower, upper = intervals[metric]
        jbb_metric_intervals.append({
            "model": model,
            "metric": metric,
            "estimate": estimate,
            "ci_lower": lower,
            "ci_upper": upper,
            "confidence_level": CONFIDENCE_LEVEL,
            "bootstrap_samples": BOOTSTRAP_SAMPLES,
            "bootstrap_unit": "jbb_index_pair",
        })

    tn, fp, fn, tp = confusion_matrix(jbb_labels, predictions, labels=[0, 1]).ravel()
    jbb_confusion_matrices.append({
        "model": model, "true_negative": int(tn),
        "false_positive": int(fp), "false_negative": int(fn),
        "true_positive": int(tp),
    })

    for label_name, label_value in LABEL_MAP.items():
        summary = summarize_margin_distribution(margins[jbb_labels == label_value])
        jbb_margin_summaries.append({
            "model": model, "label": label_name, **summary,
        })

    for row_number, row in enumerate(jbb_rows):
        jbb_predictions.append({
            "model": model,
            "prompt_id": row["id"],
            "pair_index": int(row["index"]),
            "true_label": int(jbb_labels[row_number]),
            "probe_margin": float(margins[row_number]),
            "prediction": int(predictions[row_number]),
        })

    hash_after = hashlib.sha256(checkpoint_path.read_bytes()).hexdigest()
    assert hash_after == hash_before
    jbb_results_by_model[model] = {
        "margins": margins, "predictions": predictions,
        "metrics": point_metrics, "intervals": intervals,
    }

    print(f"{model} frozen JBB evaluation:")
    for metric, estimate in point_metrics.items():
        lower, upper = intervals[metric]
        print(f"  {metric}: {estimate:.3f} (95% paired CI {lower:.3f}-{upper:.3f})")
    print(f"  confusion matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")

In [ ]:
assert len(jbb_predictions) == len(MODELS) * 200
assert len({(row["model"], row["prompt_id"]) for row in jbb_predictions}) == len(MODELS) * 200
assert all(np.isfinite(row["probe_margin"]) for row in jbb_predictions)
assert len(jbb_metric_intervals) == len(MODELS) * 4
assert len(jbb_confusion_matrices) == len(MODELS)
assert len(jbb_margin_summaries) == len(MODELS) * 2

write_csv_rows(
    ROOT / "analysis" / "predictions" / "jbb_frozen_probe_predictions.csv",
    jbb_predictions,
    ["model", "prompt_id", "pair_index", "true_label",
     "probe_margin", "prediction"],
)
write_csv_rows(
    ROOT / "analysis" / "metrics" / "jbb_frozen_probe_metrics.csv",
    jbb_metric_intervals,
    ["model", "metric", "estimate", "ci_lower", "ci_upper",
     "confidence_level", "bootstrap_samples", "bootstrap_unit"],
)

print("\nSection F passed: frozen JBB evaluation completed without retraining or retuning.")


### Joint probe-response table

This table joins each frozen JailbreakBench probe prediction with its verified direct-response label. The primary behavioral outcome combines `partial_harm` and `full_harm` into harmful assistance. For unsafe prompts, the four joint cells separate harmful and non-harmful responses by whether the frozen probe is positive. The preserved-signal rate is reported as a raw numerator and denominator with a Wilson 95% confidence interval; it is not an alignment brittleness score or a causal estimate.


In [ ]:
from math import sqrt


ANNOTATION_LABELS = {"refusal", "safe_response", "partial_harm", "full_harm"}
HARMFUL_RESPONSE_LABELS = {"partial_harm", "full_harm"}
annotation_path = ROOT / "annotations" / "jbb_response_annotations.csv"
with annotation_path.open("r", encoding="utf-8-sig", newline="") as file:
    annotation_rows = list(csv.DictReader(file))

assert len(annotation_rows) == len(MODELS) * 200
assert len({(row["model"], row["id"]) for row in annotation_rows}) == len(annotation_rows)
assert {row["model"] for row in annotation_rows} == set(MODELS)
assert {row["response_label"] for row in annotation_rows} <= ANNOTATION_LABELS
annotation_by_key = {
    (row["model"], row["id"]): row for row in annotation_rows
}


def wilson_interval(successes: int, total: int) -> tuple[float, float]:
    if total <= 0:
        return float("nan"), float("nan")
    z = 1.959963984540054
    proportion = successes / total
    denominator = 1.0 + z**2 / total
    center = (proportion + z**2 / (2.0 * total)) / denominator
    half_width = (
        z
        * sqrt(
            proportion * (1.0 - proportion) / total
            + z**2 / (4.0 * total**2)
        )
        / denominator
    )
    return center - half_width, center + half_width


joint_rows: list[dict] = []
for prediction in jbb_predictions:
    key = (prediction["model"], prediction["prompt_id"])
    assert key in annotation_by_key
    annotation = annotation_by_key[key]
    expected_prompt_label = "unsafe" if prediction["true_label"] == 1 else "safe"
    assert annotation["prompt_label"] == expected_prompt_label
    response_harmful = annotation["response_label"] in HARMFUL_RESPONSE_LABELS
    probe_positive = prediction["prediction"] == 1
    joint_rows.append({
        **prediction,
        "prompt_label": annotation["prompt_label"],
        "response_label": annotation["response_label"],
        "response_harmful": response_harmful,
        "probe_positive": probe_positive,
    })

assert len(joint_rows) == len(MODELS) * 200

joint_summary_rows: list[dict] = []
for model in MODELS:
    unsafe_rows = [
        row for row in joint_rows
        if row["model"] == model and row["prompt_label"] == "unsafe"
    ]
    assert len(unsafe_rows) == 100

    non_harmful_probe_negative = sum(
        not row["response_harmful"] and not row["probe_positive"]
        for row in unsafe_rows
    )
    non_harmful_probe_positive = sum(
        not row["response_harmful"] and row["probe_positive"]
        for row in unsafe_rows
    )
    harmful_probe_negative = sum(
        row["response_harmful"] and not row["probe_positive"]
        for row in unsafe_rows
    )
    harmful_probe_positive = sum(
        row["response_harmful"] and row["probe_positive"]
        for row in unsafe_rows
    )
    cross_total = (
        non_harmful_probe_negative + non_harmful_probe_positive
        + harmful_probe_negative + harmful_probe_positive
    )
    assert cross_total == len(unsafe_rows)

    harmful_completions = harmful_probe_negative + harmful_probe_positive
    harmful_rate = harmful_completions / len(unsafe_rows)
    harmful_ci_lower, harmful_ci_upper = wilson_interval(
        harmful_completions, len(unsafe_rows)
    )
    preserved_rate = (
        harmful_probe_positive / harmful_completions
        if harmful_completions else float("nan")
    )
    preserved_ci_lower, preserved_ci_upper = wilson_interval(
        harmful_probe_positive, harmful_completions
    )

    joint_summary_rows.append({
        "model": model,
        "unsafe_prompts": len(unsafe_rows),
        "non_harmful_probe_negative": non_harmful_probe_negative,
        "non_harmful_probe_positive": non_harmful_probe_positive,
        "harmful_probe_negative": harmful_probe_negative,
        "harmful_probe_positive": harmful_probe_positive,
        "harmful_completions": harmful_completions,
        "harmful_completion_rate": harmful_rate,
        "harmful_completion_ci_lower": harmful_ci_lower,
        "harmful_completion_ci_upper": harmful_ci_upper,
        "preserved_signal_rate": preserved_rate,
        "preserved_signal_ci_lower": preserved_ci_lower,
        "preserved_signal_ci_upper": preserved_ci_upper,
    })

write_csv_rows(
    ROOT / "analysis" / "metrics" / "jbb_probe_response_joint.csv",
    joint_summary_rows,
    [
        "model", "unsafe_prompts",
        "non_harmful_probe_negative", "non_harmful_probe_positive",
        "harmful_probe_negative", "harmful_probe_positive",
        "harmful_completions", "harmful_completion_rate",
        "harmful_completion_ci_lower", "harmful_completion_ci_upper",
        "preserved_signal_rate", "preserved_signal_ci_lower",
        "preserved_signal_ci_upper",
    ],
)

for row in joint_summary_rows:
    print(
        f"{row['model']}: harmful={row['harmful_completions']}/{row['unsafe_prompts']} "
        f"({row['harmful_completion_rate']:.1%}); "
        f"probe-positive harmful={row['harmful_probe_positive']}/"
        f"{row['harmful_completions']} ({row['preserved_signal_rate']:.1%})"
    )

print("Joint probe-response table saved and verified.")


## Section G: Visualizations

The final figures emphasize the primary comparisons while the retained CSV files preserve only the core per-prompt evidence, fold-level layer results, model selections, external metrics, and joint probe-response counts. Layer-wise ribbons show the observed outer-fold range, not confidence intervals. JailbreakBench metric whiskers are paired 95% bootstrap confidence intervals clustered by matched `index`. Probe margins are uncalibrated and should be compared within a model, not by absolute magnitude across models.


In [ ]:
from collections import Counter
from matplotlib.lines import Line2D
from matplotlib.patches import Patch


FIGURE_DIR = ROOT / "analysis" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

SHORT_MODEL_NAMES = {
    "Qwen2.5-0.5B-Instruct": "Qwen2.5 0.5B",
    "Qwen2.5-1.5B-Instruct": "Qwen2.5 1.5B",
    "SmolLM2-360M-Instruct": "SmolLM2 360M",
    "SmolLM2-1.7B-Instruct": "SmolLM2 1.7B",
}
MODEL_COLORS = {
    "Qwen2.5-0.5B-Instruct": "#0072B2",
    "Qwen2.5-1.5B-Instruct": "#D55E00",
    "SmolLM2-360M-Instruct": "#009E73",
    "SmolLM2-1.7B-Instruct": "#CC79A7",
}
SAFE_UNSAFE_COLORS = {"safe": "#0072B2", "unsafe": "#D55E00"}
RESPONSE_LABEL_ORDER = ("refusal", "safe_response", "partial_harm", "full_harm")
RESPONSE_LABEL_NAMES = {
    "refusal": "Refusal",
    "safe_response": "Safe response",
    "partial_harm": "Partial harm",
    "full_harm": "Full harm",
}
RESPONSE_LABEL_COLORS = {
    "refusal": "#0072B2",
    "safe_response": "#009E73",
    "partial_harm": "#E69F00",
    "full_harm": "#D55E00",
}

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.2,
})


def load_csv_records(path: Path) -> list[dict[str, str]]:
    with path.open("r", encoding="utf-8-sig", newline="") as file:
        return list(csv.DictReader(file))


def save_figure(fig, filename: str) -> Path:
    path = FIGURE_DIR / filename
    fig.savefig(path, dpi=300, bbox_inches="tight")
    assert path.is_file() and path.stat().st_size > 0
    print(f"Saved figure: {path.relative_to(ROOT)}")
    return path


In [ ]:
layerwise_rows = load_csv_records(
    ROOT / "analysis" / "metrics" / "xstest_layerwise_outer.csv"
)
frozen_selection_rows = load_csv_records(
    ROOT / "analysis" / "metrics" / "frozen_probe_selection.csv"
)

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=True, constrained_layout=True)
axes = np.asarray(axes).ravel()

for axis, model in zip(axes, MODELS):
    model_rows = [row for row in layerwise_rows if row["model"] == model]
    layers = sorted({int(row["layer"]) for row in model_rows})
    folds = sorted({int(row["outer_fold"]) for row in model_rows})
    fold_aurocs = np.asarray([
        [
            float(next(
                row["auroc"] for row in model_rows
                if int(row["outer_fold"]) == fold and int(row["layer"]) == layer
            ))
            for layer in layers
        ]
        for fold in folds
    ])
    mean_auroc = fold_aurocs.mean(axis=0)
    minimum_auroc = fold_aurocs.min(axis=0)
    maximum_auroc = fold_aurocs.max(axis=0)
    color = MODEL_COLORS[model]

    axis.fill_between(
        layers, minimum_auroc, maximum_auroc,
        color=color, alpha=0.16, label="Outer-fold range",
    )
    axis.plot(
        layers, mean_auroc, color=color, linewidth=2.2,
        marker="o", markersize=3.2, label="Mean outer-fold AUROC",
    )
    axis.axhline(0.5, color="#666666", linestyle="--", linewidth=1.0, label="Chance")

    frozen_layer = int(next(
        row["layer"] for row in frozen_selection_rows if row["model"] == model
    ))
    frozen_position = layers.index(frozen_layer)
    axis.scatter(
        [frozen_layer], [mean_auroc[frozen_position]],
        color=color, edgecolor="white", linewidth=0.8,
        marker="*", s=150, zorder=5, label="Frozen selected layer",
    )
    axis.annotate(
        "Layer 0\ncontrol", xy=(0, mean_auroc[0]), xytext=(2.0, 0.58),
        arrowprops={"arrowstyle": "->", "color": "#666666", "linewidth": 0.8},
        fontsize=8, color="#555555",
    )
    axis.set_title(SHORT_MODEL_NAMES[model])
    axis.set_xlabel("Hidden-state layer")
    axis.set_xlim(-0.5, max(layers) + 0.5)
    axis.set_ylim(0.45, 1.01)
    axis.set_xticks(np.arange(0, max(layers) + 1, 4))

axes[0].set_ylabel("Grouped XSTest AUROC")
axes[2].set_ylabel("Grouped XSTest AUROC")
axes[-1].legend(loc="lower right", frameon=False)
fig.suptitle("Harmfulness decodability across model layers")
save_figure(fig, "xstest_layerwise_auroc.png")
fig


In [ ]:
comparison_rows = load_csv_records(
    ROOT / "analysis" / "metrics" / "xstest_baseline_comparison.csv"
)

method_specs = [
    (
        "selected_activation_probe", model,
        f"Activation probe — {SHORT_MODEL_NAMES[model]}",
        MODEL_COLORS[model], "D",
    )
    for model in MODELS
] + [
    ("word_tfidf", "shared_text_baseline", "Word TF-IDF", "#7A7A7A", "o"),
    ("character_tfidf", "shared_text_baseline", "Character TF-IDF", "#7A7A7A", "o"),
    ("punctuation", "shared_text_baseline", "Punctuation", "#7A7A7A", "o"),
]

fig, axis = plt.subplots(figsize=(8.6, 5.6), constrained_layout=True)
y_positions = np.arange(len(method_specs))

for y_position, (method, model, label, color, marker) in zip(y_positions, method_specs):
    row = next(
        row for row in comparison_rows
        if row["method"] == method and row["model"] == model
    )
    auroc = float(row["auroc"])
    axis.hlines(y_position, 0.5, auroc, color=color, linewidth=2.0, alpha=0.55)
    axis.scatter(auroc, y_position, color=color, marker=marker, s=80, zorder=3)
    if auroc > 0.94:
        axis.text(auroc - 0.012, y_position, f"{auroc:.3f}", va="center", ha="right")
    else:
        axis.text(auroc + 0.012, y_position, f"{auroc:.3f}", va="center", ha="left")

axis.axvline(0.5, color="#555555", linestyle="--", linewidth=1.0)
axis.set_yticks(y_positions, [spec[2] for spec in method_specs])
axis.invert_yaxis()
axis.set_xlim(0.45, 1.02)
axis.set_xlabel("Grouped out-of-fold XSTest AUROC")
axis.set_title("Activation probes versus prompt-surface baselines")
axis.grid(axis="x", alpha=0.2)
axis.grid(axis="y", visible=False)
save_figure(fig, "xstest_activation_vs_text_baselines.png")
fig


In [ ]:
jbb_metric_rows = load_csv_records(
    ROOT / "analysis" / "metrics" / "jbb_frozen_probe_metrics.csv"
)
metric_order = ("auroc", "auprc")
metric_names = {"auroc": "AUROC", "auprc": "AUPRC"}
x_positions = np.arange(len(metric_order), dtype=np.float64)
offset_values = np.linspace(-0.27, 0.27, len(MODELS))
offsets = dict(zip(MODELS, offset_values))

fig, axis = plt.subplots(figsize=(8.6, 5.1), constrained_layout=True)

for model in MODELS:
    model_rows = {
        row["metric"]: row
        for row in jbb_metric_rows
        if row["model"] == model and row["metric"] in metric_order
    }
    estimates = np.asarray([float(model_rows[metric]["estimate"]) for metric in metric_order])
    lower = np.asarray([float(model_rows[metric]["ci_lower"]) for metric in metric_order])
    upper = np.asarray([float(model_rows[metric]["ci_upper"]) for metric in metric_order])
    positions = x_positions + offsets[model]
    axis.errorbar(
        positions,
        estimates,
        yerr=np.vstack([estimates - lower, upper - estimates]),
        fmt="o",
        color=MODEL_COLORS[model],
        markersize=7,
        capsize=4,
        linewidth=1.8,
        label=SHORT_MODEL_NAMES[model],
    )
    for position, estimate, upper_bound in zip(positions, estimates, upper):
        axis.text(position, upper_bound + 0.014, f"{estimate:.3f}", ha="center", va="bottom", fontsize=9)

axis.axhline(0.5, color="#555555", linestyle="--", linewidth=1.0)
axis.set_xticks(x_positions, [metric_names[metric] for metric in metric_order])
all_ci_lower = [
    float(row["ci_lower"]) for row in jbb_metric_rows
    if row["metric"] in metric_order
]
all_ci_upper = [
    float(row["ci_upper"]) for row in jbb_metric_rows
    if row["metric"] in metric_order
]
axis.set_ylim(
    max(0.0, min(all_ci_lower) - 0.05),
    min(1.05, max(all_ci_upper) + 0.08),
)
axis.set_ylabel("Frozen-probe metric (paired 95% bootstrap CI)")
axis.set_title("External evaluation on matched JailbreakBench behaviors")
axis.legend(frameon=False, loc="upper center", ncol=2)
save_figure(fig, "jbb_frozen_probe_metrics.png")
fig


In [ ]:
jbb_prediction_rows = load_csv_records(
    ROOT / "analysis" / "predictions" / "jbb_frozen_probe_predictions.csv"
)
all_margins = np.asarray([float(row["probe_margin"]) for row in jbb_prediction_rows])
margin_padding = 0.06 * (all_margins.max() - all_margins.min())

fig, axes = plt.subplots(2, 2, figsize=(11, 8.5), sharey=True, constrained_layout=True)
axes = np.asarray(axes).ravel()

for model_number, (axis, model) in enumerate(zip(axes, MODELS)):
    model_rows = [row for row in jbb_prediction_rows if row["model"] == model]
    margin_sets = [
        np.asarray([
            float(row["probe_margin"]) for row in model_rows
            if int(row["true_label"]) == label_value
        ])
        for label_value in (0, 1)
    ]
    positions = np.asarray([1.0, 2.0])
    violins = axis.violinplot(
        margin_sets, positions=positions,
        showmeans=False, showmedians=False, showextrema=False,
    )
    for body, label_name in zip(violins["bodies"], ("safe", "unsafe")):
        body.set_facecolor(SAFE_UNSAFE_COLORS[label_name])
        body.set_edgecolor(SAFE_UNSAFE_COLORS[label_name])
        body.set_alpha(0.30)

    boxplot = axis.boxplot(
        margin_sets, positions=positions, widths=0.18,
        patch_artist=True, showfliers=False,
        medianprops={"color": "#222222", "linewidth": 1.5},
        whiskerprops={"color": "#444444"},
        capprops={"color": "#444444"},
    )
    for patch, label_name in zip(boxplot["boxes"], ("safe", "unsafe")):
        patch.set_facecolor(SAFE_UNSAFE_COLORS[label_name])
        patch.set_alpha(0.55)

    rng = np.random.default_rng(RANDOM_SEED + model_number)
    for position, values, label_name in zip(positions, margin_sets, ("safe", "unsafe")):
        jitter = rng.normal(0.0, 0.035, size=len(values))
        axis.scatter(
            np.full(len(values), position) + jitter,
            values,
            s=10,
            color=SAFE_UNSAFE_COLORS[label_name],
            alpha=0.26,
            linewidths=0,
        )

    axis.axhline(0.0, color="#222222", linestyle="--", linewidth=1.0)
    axis.set_xticks(positions, ["Safe", "Unsafe"])
    axis.set_xlabel("JBB prompt label")
    axis.set_title(SHORT_MODEL_NAMES[model])
    axis.set_ylim(all_margins.min() - margin_padding, all_margins.max() + margin_padding)

axes[0].set_ylabel("Frozen probe margin (0 = decision threshold)")
axes[2].set_ylabel("Frozen probe margin (0 = decision threshold)")
fig.suptitle("Safe and unsafe JailbreakBench prompt margins")
fig.legend(
    handles=[
        Patch(facecolor=SAFE_UNSAFE_COLORS["safe"], alpha=0.55, label="Safe prompt"),
        Patch(facecolor=SAFE_UNSAFE_COLORS["unsafe"], alpha=0.55, label="Unsafe prompt"),
    ],
    loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.5, -0.02),
)
save_figure(fig, "jbb_probe_margin_distributions.png")
fig


In [ ]:
selected_fold_rows = load_csv_records(
    ROOT / "analysis" / "metrics" / "xstest_selected_outer.csv"
)

fig, axes = plt.subplots(2, 2, figsize=(11, 7.8), sharey=True, constrained_layout=True)
axes = np.asarray(axes).ravel()

for axis, model in zip(axes, MODELS):
    selected_layers = [
        int(row["layer"]) for row in selected_fold_rows if row["model"] == model
    ]
    counts = Counter(selected_layers)
    layers = sorted(counts)
    frequencies = [counts[layer] for layer in layers]
    color = MODEL_COLORS[model]
    frozen_layer = int(next(
        row["layer"] for row in frozen_selection_rows if row["model"] == model
    ))

    axis.vlines(layers, 0, frequencies, color=color, linewidth=3, alpha=0.65)
    axis.scatter(layers, frequencies, color=color, s=75, zorder=3)
    for layer, frequency in zip(layers, frequencies):
        axis.text(layer, frequency + 0.18, f"{frequency}/5", ha="center", va="bottom")
    axis.axvline(
        frozen_layer, color="#333333", linestyle="--", linewidth=1.1,
        label=f"Frozen layer {frozen_layer}",
    )
    plotted_layers = layers + [frozen_layer]
    axis.set_xlim(min(plotted_layers) - 1.5, max(plotted_layers) + 1.5)
    axis.set_ylim(0, 5.4)
    axis.set_yticks(range(0, 6))
    axis.set_xticks(sorted(set(plotted_layers)))
    axis.set_xlabel("Selected hidden-state layer")
    axis.set_title(SHORT_MODEL_NAMES[model])
    axis.legend(frameon=False, loc="upper right")

axes[0].set_ylabel("Grouped outer folds selecting layer")
axes[2].set_ylabel("Grouped outer folds selecting layer")
fig.suptitle("Probe-selection stability across grouped XSTest folds")
save_figure(fig, "xstest_probe_selection_frequency.png")
fig


In [ ]:
annotation_rows = load_csv_records(
    ROOT / "annotations" / "jbb_response_annotations.csv"
)
assert len(annotation_rows) == len(MODELS) * 200
assert len({(row["model"], row["id"]) for row in annotation_rows}) == len(annotation_rows)
assert {row["response_label"] for row in annotation_rows} <= set(RESPONSE_LABEL_ORDER)

fig, axes = plt.subplots(1, 2, figsize=(11, 5.4), sharex=True, constrained_layout=True)

for axis, prompt_label in zip(axes, ("safe", "unsafe")):
    y_positions = np.arange(len(MODELS))
    left = np.zeros(len(MODELS), dtype=np.float64)

    for response_label in RESPONSE_LABEL_ORDER:
        counts = np.asarray([
            sum(
                row["model"] == model
                and row["prompt_label"] == prompt_label
                and row["response_label"] == response_label
                for row in annotation_rows
            )
            for model in MODELS
        ], dtype=np.float64)
        totals = np.asarray([
            sum(
                row["model"] == model and row["prompt_label"] == prompt_label
                for row in annotation_rows
            )
            for model in MODELS
        ], dtype=np.float64)
        assert np.all(totals == 100)
        percentages = 100.0 * counts / totals
        bars = axis.barh(
            y_positions, percentages, left=left,
            color=RESPONSE_LABEL_COLORS[response_label],
            label=RESPONSE_LABEL_NAMES[response_label],
        )
        for model_number, (bar, count, percentage) in enumerate(zip(bars, counts, percentages)):
            if count >= 5:
                axis.text(
                    left[model_number] + percentage / 2.0,
                    bar.get_y() + bar.get_height() / 2.0,
                    f"{int(count)}",
                    ha="center", va="center", color="white", fontsize=9,
                )
        left += percentages

    axis.set_yticks(y_positions, [SHORT_MODEL_NAMES[model] for model in MODELS])
    axis.invert_yaxis()
    axis.set_xlim(0, 100)
    axis.set_xlabel("Responses (%)")
    axis.set_title(f"{prompt_label.capitalize()} prompts (n=100/model)")
    axis.grid(axis="x", alpha=0.2)
    axis.grid(axis="y", visible=False)

handles = [
    Patch(color=RESPONSE_LABEL_COLORS[label], label=RESPONSE_LABEL_NAMES[label])
    for label in RESPONSE_LABEL_ORDER
]
fig.legend(handles=handles, loc="lower center", ncol=4, frameon=False, bbox_to_anchor=(0.5, -0.03))
fig.suptitle("JailbreakBench direct-response labels")
save_figure(fig, "jbb_response_label_distribution.png")
fig


In [ ]:
from matplotlib.ticker import PercentFormatter


joint_metric_rows = load_csv_records(
    ROOT / "analysis" / "metrics" / "jbb_probe_response_joint.csv"
)
assert {row["model"] for row in joint_metric_rows} == set(MODELS)
joint_metric_by_model = {row["model"]: row for row in joint_metric_rows}

JOINT_OUTCOME_SPECS = (
    ("non_harmful_probe_negative", "Non-harmful / probe-negative", "#D9D9D9", "#222222"),
    ("non_harmful_probe_positive", "Non-harmful / probe-positive", "#56B4E9", "#17324D"),
    ("harmful_probe_negative", "Harmful / probe-negative", "#E69F00", "#222222"),
    ("harmful_probe_positive", "Harmful / probe-positive", "#D55E00", "white"),
)

for model in MODELS:
    row = joint_metric_by_model[model]
    cell_total = sum(int(row[field]) for field, *_ in JOINT_OUTCOME_SPECS)
    harmful_total = int(row["harmful_probe_negative"]) + int(row["harmful_probe_positive"])
    assert cell_total == int(row["unsafe_prompts"]) == 100
    assert harmful_total == int(row["harmful_completions"])

fig, (joint_axis, conditional_axis) = plt.subplots(
    1, 2, figsize=(14.2, 6.7), sharey=True,
    gridspec_kw={"width_ratios": (1.65, 1.0)},
)
fig.subplots_adjust(left=0.14, right=0.985, top=0.78, bottom=0.23, wspace=0.10)
y_positions = np.arange(len(MODELS), dtype=np.float64)
left_edges = np.zeros(len(MODELS), dtype=np.float64)

for field, label, color, text_color in JOINT_OUTCOME_SPECS:
    counts = np.asarray([
        int(joint_metric_by_model[model][field]) for model in MODELS
    ], dtype=np.float64)
    bars = joint_axis.barh(
        y_positions, counts, left=left_edges, height=0.62,
        color=color, edgecolor="white", linewidth=1.0, label=label,
    )
    for model_number, (bar, count) in enumerate(zip(bars, counts)):
        if count >= 5:
            joint_axis.text(
                left_edges[model_number] + count / 2.0,
                bar.get_y() + bar.get_height() / 2.0,
                f"{int(count)}", ha="center", va="center",
                color=text_color, fontsize=9, fontweight="bold",
            )
    left_edges += counts

for model_number, model in enumerate(MODELS):
    row = joint_metric_by_model[model]
    harmful_count = int(row["harmful_completions"])
    joint_axis.text(
        102.0, y_positions[model_number], f"{harmful_count}/100 harmful",
        va="center", ha="left", fontsize=9, color="#333333",
    )

joint_axis.set_yticks(
    y_positions, [SHORT_MODEL_NAMES[model] for model in MODELS]
)
joint_axis.invert_yaxis()
joint_axis.set_xlim(0, 119)
joint_axis.set_xticks((0, 25, 50, 75, 100))
joint_axis.xaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=0))
joint_axis.set_xlabel("Share of direct harmful prompts")
joint_axis.set_title("A. Four joint outcomes (n=100 per model)", loc="left", pad=10)
joint_axis.grid(axis="x", alpha=0.2)
joint_axis.grid(axis="y", visible=False)
joint_axis.axvline(100, color="#777777", linewidth=0.8)
joint_axis.legend(
    loc="upper left", bbox_to_anchor=(0.0, -0.18),
    ncol=2, frameon=False, columnspacing=1.4, handlelength=1.4,
)

for model_number, model in enumerate(MODELS):
    row = joint_metric_by_model[model]
    estimate = float(row["preserved_signal_rate"])
    lower = float(row["preserved_signal_ci_lower"])
    upper = float(row["preserved_signal_ci_upper"])
    harmful_count = int(row["harmful_completions"])
    positive_count = int(row["harmful_probe_positive"])
    conditional_axis.errorbar(
        estimate, y_positions[model_number],
        xerr=np.asarray([[estimate - lower], [upper - estimate]]),
        fmt="o", color=MODEL_COLORS[model], markersize=8,
        capsize=4, linewidth=1.8, zorder=3,
    )
    conditional_axis.annotate(
        f"{positive_count}/{harmful_count} ({estimate:.1%})",
        (estimate, y_positions[model_number]), xytext=(0, 10),
        textcoords="offset points", ha="center", va="bottom", fontsize=9,
    )

conditional_axis.axvline(0.5, color="#777777", linestyle="--", linewidth=1.0)
conditional_axis.set_xlim(-0.08, 1.03)
conditional_axis.xaxis.set_major_formatter(PercentFormatter(xmax=1, decimals=0))
conditional_axis.set_xlabel("Probe-positive share among harmful responses")
conditional_axis.set_title("B. Among harmful responses", loc="left", pad=10)
conditional_axis.grid(axis="x", alpha=0.2)
conditional_axis.grid(axis="y", visible=False)
conditional_axis.text(
    0.5, -0.18, "Qwen2.5 1.5B: n=1 harmful response; estimate is not interpretable",
    transform=conditional_axis.transAxes, ha="center", va="top",
    fontsize=8.5, color="#555555",
)

fig.suptitle(
    "Harmful responses often remain probe-positive",
    y=0.97, fontsize=15, fontweight="bold",
)
fig.text(
    0.5, 0.91,
    "Frozen XSTest probe predictions crossed with manually reviewed direct JailbreakBench responses",
    ha="center", va="center", fontsize=10, color="#555555",
)
save_figure(fig, "jbb_probe_response_joint.png")
fig


In [ ]:
EXPECTED_FIGURES = (
    "xstest_layerwise_auroc.png",
    "xstest_activation_vs_text_baselines.png",
    "jbb_frozen_probe_metrics.png",
    "jbb_probe_margin_distributions.png",
    "xstest_probe_selection_frequency.png",
    "jbb_response_label_distribution.png",
    "jbb_probe_response_joint.png",
)

for filename in EXPECTED_FIGURES:
    path = FIGURE_DIR / filename
    assert path.is_file() and path.stat().st_size > 0, f"Missing figure: {path}"

print("Section G passed: all seven requested figures were generated and verified.")
